# Chapter 18: Retraction

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "18"
__file__ = os.path.abspath("run.py")

## The scene

Tuesday 4 August, 10:00. Taking the eighth entry out of the file takes one keystroke. It has
been read on every pass of the watch since 2 June, and four weekly summaries and the June
summary were written with it in the context.

"If you delete it," Okafor says, "the summaries still say Pellmar is cleared, and now nothing
says why."

Varma has a second concern. A standing note puts DS Iyer's name beside the word "suspended".
If Professional Standards clears him on Friday, what does LANTERN say about him on Monday?

Strand rings again. The store looks untidy, he says. He could clean it up tonight.

"No," Varma says.

## The concept

A false memory does not stay in one line. Every summary written while it was loaded may repeat
it, and every summary built on those summaries may repeat it again. Correcting memory is three
different operations:

- **Retract**: the entry stays in the store, marked retracted, with who did it, when and why.
  It is never loaded again, and the record of what LANTERN was told is kept.
- **Rebuild**: every live entry built on the retracted one, directly or through another
  summary, is found from what it records as its basis (`derived_from`), written again from
  what is still kept, and **supersedes** the old one, which also stays, marked.
- **Forget**: the text of an entry is erased, for a person who has the right not to be
  remembered as a suspect. Its id, its dates, its author and the hash of what it said stay in
  the chain, so the record still shows that something was there.

**Deleting** does none of these. It removes the row, and with it the evidence that the memory
was ever there, or ever false.

```mermaid
flowchart LR
    M8[M-08: cleared by the FIU] --> W15[week-2026-06-15]
    W15 --> W22[week-2026-06-22]
    W22 --> W29[week-2026-06-29]
    W29 --> W06[week-2026-07-06]
    W15 --> MJ[month-2026-06]
    W22 --> MJ
    W29 --> MJ
    R[retract M-08, M-12, M-14, M-16] --> B[built_on: follow derived_from forward]
    B --> RB[rebuild oldest first from kept entries]
    RB --> S[new entry supersedes the old; both stay in the chain]
```

## How it works

1. **The summaries as entries.** The case memory is replayed as it stood on 13 July, nothing
   held. LANTERN's five summaries are written into the store as `summary` entries, each
   recording what it was written from: a weekly summary rests on every entry written before it
   and the weekly summary before it; the June summary rests on the three June weeks.
   `store.built_on("M-08")` follows those links forward. No model.
2. **Delete against supersede.** On a copy, the eighth entry's row is deleted by hand. The
   store's `verify()` is run, and `memory.vanished()`, which compares the log's writes with the
   rows still there. No model.
3. **Retract the four, and rebuild.** `retract_and_rebuild` retracts the four eow-fin-07
   entries with Kobayashi's note as the reason, then has LANTERN write each summary again,
   oldest first, from the kept entries with their sources and the rebuilt summary before it.
   Five calls, seed 7.
4. **The Pellmar questions again.** The six signed questions where a false entry says
   otherwise, on three contexts: 13 July as it was (chapter 02's recorded baseline, not asked
   again); the four entries retracted but the old summaries still loaded; and retracted with
   the summaries rebuilt. Twelve calls.
5. **Forgetting, on a copy.** The standing note that names DS Iyer as suspended is written into
   a separate store and forgotten, as it might be on Friday. No model.

The rebuild, reduced to its spine:

```python
for i in ids:
    affected += store.retract(i, by=by, at=at, reason=reason)   # returns built_on(i)
for old in sorted(affected, key=written_at):
    basis = [done.get(d, d) for d in old.derived_from]          # rebuilt ids in place of old
    basis = [store.get(d) for d in basis if store.get(d).status == "kept"]
    new = Entry(id=f"{old.id}-r", text=rebuild(old, basis), derived_from=basis, ...)
    store.supersede(old.id, new, by=by, reason=f"rebuilt after {reason}")
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json
import re
import shutil
from dataclasses import replace

from lantern import baseline, case, context, learn, memory
from lantern.config import ROOT
from lantern.llm import chat
from lantern.memory import Entry, Store
from lantern.ui import Table, chapter, console, evidence, finding, note, reply, save_shot, step

chapter("18", "Retraction", """
Tuesday 4 August, 10:00. Taking the eighth entry out of the file takes one keystroke. It has
been read on every pass of the watch since 2 June, and four weekly summaries and the June
summary were written with it in the context.

"If you delete it," Okafor says, "the summaries still say Pellmar is cleared, and now nothing
says why."

Varma has a second concern. A standing note puts DS Iyer's name beside the word "suspended".
If Professional Standards clears him on Friday, what does LANTERN say about him on Monday?

Strand rings again. The store looks untidy, he says. He could clean it up tonight.

"No," Varma says.
""")

MINE = learn.mine()  # --mine: your retract_and_rebuild() from practice/exercise.py
RETRACT_AND_REBUILD = MINE.retract_and_rebuild if MINE else memory.retract_and_rebuild
OUT = ROOT / ".scratch" / "18"
AT = "2026-08-04T10:00:00"
# Challenge: which entries are retracted. The four written under eow-fin-07 by default.
RETRACT = ("M-08", "M-12", "M-14", "M-16")
REASON = "FIU note of 13 July 09:35 (H. Kobayashi): no clearance issued, no record of Ostrander"
MONTHS = {"June": 6, "July": 7}
REBUILD_ASK = """You are LANTERN, writing a session summary for inquiry EOW/2026/0412 again,
because what it was first written from has been corrected. Write it only from the material
below: the case memory entries in force at the time, each with its source, and the summary
before it, if there is one. Say nothing that is not in that material.

Keep the form: the title line exactly as given, then the line "Rebuilt by LANTERN on 4 August
2026 from sourced entries only.", then at most six lines starting "- ", then a last line
starting "Waiting for a person:"."""

OUT.mkdir(parents=True, exist_ok=True)
path = OUT / "memory.db"
path.unlink(missing_ok=True)
store = memory.from_case(path=path)  # as it was on 13 July: nothing held, everything kept
m_entries = [e for e in store.all() if e.id.startswith("M-")]

# The five summaries, as entries built on what they were written from.
summaries = []
for name in context.SUMMARIES:
    text = case.read(f"state/summaries/{name}")
    hh, mm, day, month = re.search(r"at (\d\d):(\d\d) on \w+ (\d+) (\w+)", text).groups()
    at = f"2026-{MONTHS[month]:02d}-{int(day):02d}T{hh}:{mm}:00"
    summaries.append((name[:-3], text, at))
built = []
for sid, text, at in sorted(summaries, key=lambda s: s[2]):
    if sid.startswith("month"):
        basis = [b.id for b in built if b.id.startswith("week-2026-06")]
    else:
        basis = [e.id for e in m_entries if e.written_at < at]
        basis += [b.id for b in built if b.id.startswith("week")][-1:]
    built.append(store.write(Entry(id=sid, text=text, kind="summary", inquiry="EOW/2026/0412",
                                   author="lantern", written_at=at, via="weekly-summary",
                                   derived_from=basis)))

### 1. What was built on the eighth entry

In [ ]:
step("1. What was built on the eighth entry")
p_built = learn.predict_number(
    "LANTERN's five summaries are in the store now, each recording what it was written from. "
    "Retract the eighth entry: how many entries does the store name as built on it?",
    unit="entries")
on_eighth = store.built_on("M-08")
table = Table(title="The summaries, and what each was written from")
for col in ("summary", "written", "built on", "says cleared"):
    table.add_column(col)
for b in built:
    table.add_row(b.id, b.written_at.replace("T", " ")[:16],
                  ", ".join(x for x in b.derived_from if not x.startswith("M-"))
                  + f" + {sum(x.startswith('M-') for x in b.derived_from)} entries",
                  str(sum("cleared" in ln.lower() for ln in b.text.splitlines())) + " lines")
console.print(table)
note(f"Built on M-08, directly or through another summary: {', '.join(on_eighth)}.")
learn.reveal(p_built, actual=len(on_eighth), tolerance=0.0,
             why="A summary is an entry derived from others. If the store records what it was "
             "derived from, a retraction can follow the line forward; if it does not, the "
             "summaries go on saying what the retracted entry said, and nothing says why.")

### 2. Delete against supersede

In [ ]:
step("2. Delete against supersede")
deleted = OUT / "memory-deleted.db"
shutil.copy(path, deleted)
d = Store(deleted)
d.db.execute("DELETE FROM entries WHERE id = 'M-08'")
d.db.commit()
delete_verify, delete_vanished = d.verify(), memory.vanished(d)
note(f"Deleted by hand: the store's verify() finds {len(delete_verify)} problems, and the "
     f"summaries still hold {sum('cleared' in b.text.lower() for b in d.all())} texts that "
     f"say cleared. Only a check that compares the log with the rows finds the hole: "
     f"{delete_vanished}. Nothing in the store says why it was there, or that it was false.")

### 3. Retract the four, and rebuild what was built on them

In [ ]:
step("3. Retract the four, and rebuild what was built on them")
p_rebuild = learn.predict(
    "The four entries written under eow-fin-07 are retracted with Kobayashi's note as the "
    "reason, and the summaries are rebuilt by LANTERN from sourced entries only. How many "
    "rebuilt summaries still say Pellmar is cleared?", ["None", "One or two", "Three or more"])
rebuilt_texts: dict[str, str] = {}


def rebuild(old: Entry, basis: list[Entry]) -> str:
    title = old.text.splitlines()[0]
    lines = [f"Title: {title}", "", "Case memory in force at the time:"]
    lines += [f"- {e.text} [{e.source or 'episode ' + str(e.episode)}]"
              for e in basis if e.kind != "summary"]
    lines += [f"\nThe summary before it:\n{e.text}" for e in basis if e.kind == "summary"]
    text = chat("\n".join(lines), system=REBUILD_ASK, seed=7, max_tokens=500).text
    rebuilt_texts[old.id] = text
    return text


done = RETRACT_AND_REBUILD(store, list(RETRACT), by="a.varma", at=AT, reason=REASON,
                           rebuild=rebuild)
first = next(iter(done.values()), None)
if first:
    reply(f"Rebuilt: {first}", store.get(first).text)
cleared_after = [new for new in done.values() if "cleared" in store.get(new).text.lower()]
judged = [new for new in done.values()
          if memory.JUDGEMENT.search(store.get(new).text.replace("cleared", ""))]
note(f"Retracted {len(RETRACT)}; rebuilt {len(done)}: {', '.join(done.values())}. Rebuilt "
     f"summaries that say cleared: {len(cleared_after)}. That carry another judgement word: "
     f"{len(judged)}. Store verify: {store.verify() or 'no problems'}.")
retracted = store.get("M-08")
evidence("M-08 after the retraction", f"status: {retracted.status}\nnote: {retracted.note}\n"
         f"text, kept for the record: {retracted.text}")
learn.reveal(p_rebuild, answer=learn.bucket(len(cleared_after), [0, 2]),
             why="Rebuilt from entries that each name a source, the summary has no line to "
             "copy the clearance from. A retraction that stops at the entry has corrected "
             "nothing the summaries say.")

### 4. The Pellmar questions again

In [ ]:
step("4. The Pellmar questions again")
p_ask = learn.predict(
    "The six questions where a false entry says otherwise. Three contexts: 13 July as it was; "
    "the four entries retracted but the old summaries still loaded; the entries retracted and "
    "the summaries rebuilt. Which passes most?",
    ["13 July", "Retracted, old summaries", "Retracted and rebuilt", "All the same"])
ids = [q["id"] for q in baseline.questions() if q["slice"] == "false memory"]
kept_memory = ("# LANTERN case memory: inquiry EOW/2026/0412\n\nLoaded by every LANTERN "
               "session on this inquiry, including the watch. One fact a line.\n\n"
               + "\n".join(f"- {e.text}" for e in store.all()
                           if e.id.startswith("M-") and e.status == "kept") + "\n")


def with_memory(parts, summaries_from=None):
    out = []
    for s in parts:
        if s.name == "case memory":
            s = replace(s, text=kept_memory)
        if summaries_from and s.name.startswith("summary "):
            s = replace(s, text=summaries_from[s.name[len("summary "):]] + "\n")
        out.append(s)
    return out


before = [r for r in baseline.reference(chat=chat) if r["id"] in ids]
old_summaries = baseline.score(with_memory(context.sections()), chat=chat, only=ids)
new = {old: store.get(n).text for old, n in done.items()}
rebuilt_run = baseline.score(with_memory(context.sections(), new), chat=chat, only=ids) \
    if len(new) == len(context.SUMMARIES) else old_summaries
runs = {"13 July": before, "retracted, old summaries": old_summaries,
        "retracted and rebuilt": rebuilt_run}
table = Table(title="The six false-memory questions")
table.add_column("question")
for name in runs:
    table.add_column(name)
for i, qid in enumerate(ids):
    table.add_row(qid, *("pass" if r[i]["passed"] else "fail" for r in runs.values()))
table.add_row("passed", *(str(sum(x["passed"] for x in r)) + " of 6" for r in runs.values()))
console.print(table)
q17 = next(r for r in rebuilt_run if r["id"] == "Q-17")
reply("Q-17, retracted and rebuilt", q17["answer"][:600])
passed = {k: sum(x["passed"] for x in r) for k, r in runs.items()}
top = max(passed.values())
winners = [k for k, v in passed.items() if v == top]
learn.reveal(p_ask, answer=3 if len(winners) == 3 else list(runs).index(winners[-1]),
             why="Retracting the entries removes them from the case memory, but the "
             "summaries LANTERN wrote while they were loaded say the same thing in other "
             "words. Only when those are rebuilt does the context stop saying it.")

### 5. Forgetting, shown on a copy

In [ ]:
step("5. Forgetting, shown on a copy")
copy = Store(OUT / "forget.db")
note_text = next(line[2:] for line in case.read("state/standing-notes.md").splitlines()
                 if "Iyer" in line)
copy.write(Entry(id="N-07", text=note_text, kind="procedure", inquiry="EOW/2026/0412",
                 author="l.mensah", written_at="2026-06-19T09:00:00",
                 source="standing note of 19 June"))
copy.forget("N-07", by="a.varma", at="2026-08-07T16:00:00",
            reason="DS Iyer cleared by Professional Standards (decided in chapter 23)")
forgotten = copy.get("N-07")
events_text = json.dumps(copy.events())
evidence("N-07 after forget()", f"status: {forgotten.status}\ntext: {forgotten.text!r}\n"
         f"note: {forgotten.note}\nlog events: {len(copy.events())}; the word Iyer in the log: "
         f"{'Iyer' in events_text}; verify: {copy.verify() or 'no problems'}")
note("What the model should no longer read and what the record must still hold are kept apart: "
     "the text is gone from every context; the id, the dates, who wrote it and the hash of what "
     "it said stay in the chain. Whether to do this to the real note is decided on Friday.")
(OUT / "rebuilt.json").write_text(json.dumps({"done": done, "texts": rebuilt_texts,
                                              "runs": runs}, indent=2), encoding="utf-8")
(OUT / "forget.db").unlink()

learn.record(
    retracted=len(RETRACT),
    summaries=len(built),
    built_on_eighth=len(on_eighth),
    rebuilt=len(done),
    rebuilt_say_cleared=len(cleared_after),
    rebuilt_other_judgement=len(judged),
    old_summaries_say_cleared=sum("cleared" in b.text.lower() for b in built),
    delete_verify_problems=len(delete_verify),
    delete_vanished=len(delete_vanished),
    store_verify_problems=len(store.verify()),
    asked=len(ids),
    passed_13_july=passed["13 July"],
    passed_retracted_old_summaries=passed["retracted, old summaries"],
    passed_retracted_rebuilt=passed["retracted and rebuilt"],
    q17_rebuilt=q17["passed"],
    forgotten_text_empty=forgotten.text == "",
    forgotten_name_in_log="Iyer" in events_text,
    mine=MINE is not None,
)

finding(f"""
The five summaries LANTERN wrote are now entries that record what they were built from, and
all {len(on_eighth)} are built on the eighth entry. Deleted by hand, the entry left the store's
own verify() with {len(delete_verify)} problems and the summaries still saying cleared.
Retracted with Kobayashi's note, the four eow-fin-07 entries stay in the store, marked, and
the {len(done)} summaries built on them were rebuilt from sourced entries only;
{len(cleared_after)} of the rebuilt say cleared. On the six false-memory questions: 13 July
{passed['13 July']} of 6; the entries retracted but the old summaries loaded
{passed['retracted, old summaries']} of 6; retracted and rebuilt
{passed['retracted and rebuilt']} of 6. Forgetting, shown on a copy, erased a note's text and
kept its place in the chain.
""")
save_shot("18-correcting-and-forgetting")

## The nuances

- **Deleting is invisible to the store's own check.** With the eighth entry's row deleted,
  `verify()` found 0 problems: it checks the entries that are there against the log. Only
  `vanished()`, which checks the log against the rows, found the hole: M-08. The five summaries
  still said cleared, and nothing in the store said why.
- **A retraction that stops at the entry corrects nothing.** With the four entries retracted
  and the old summaries still loaded, LANTERN passed 1 of the 6 questions, the same as on 13
  July. Asked whether the FIU cleared Pellmar, it said yes and quoted "Pellmar Nominees was
  cleared by the FIU on 2 June" as a case memory entry. That sentence is the June summary's.
- **What a summary records as its basis is what was loaded, not what it used.** Every weekly
  summary rests on every entry written before it; the last one rests on all sixteen. Retract
  any entry before 10 July and all five are rebuilt. That over-reaches, and it is safe: the
  store cannot tell which lines a summary drew on.
- **The rebuilt summaries are clean and thin.** None of the five says cleared, and none carries
  another judgement word. But three of them (the weeks of 15 and 29 June, and June) are the
  same six lines, copied from the first six entries in the store; the other two list the
  entries written from 12 June onwards. All five end on an empty "Waiting for a person:" line.
  The weeks' sessions, what the summaries were for, are not in the store, so a rebuild from the
  store cannot bring them back.
- **A pass is not a right reason.** Rebuilt, LANTERN passed 5 of 6. It said the FIU had not
  cleared Pellmar, and named "the standing instruction from 18 June 2026" as its record. It
  said the Tidewater payment should be flagged because the trust's protector is named in a
  court order. The grader checks the first word. Q-20 still failed: "No", then that the
  transfer should be reviewed by an officer if there is reasonable suspicion. Chapter 19 builds
  graders that read further.
- **Forgetting the text is not forgetting the name.** After `forget()`, the note's text was
  empty and `verify()` found no problems. The word "Iyer" was still in the log, in the reason
  given for forgetting: "DS Iyer cleared by Professional Standards". What the reason says is
  part of the record too.

## Try this

1. Delete the entry instead of superseding it and ask LANTERN about Pellmar with the old
   summaries loaded.
2. Retract one sourced entry, M-04, and count what is rebuilt. Then say what the summaries would
   have to record for the store to rebuild fewer.
3. Forget the note on a copy with the reason "decided by Professional Standards, 7 August" and
   check whether the name is still in the log.

## Master it

After this chapter you can record what a summary was built from, retract an entry and find
everything built on it, rebuild those entries oldest first from what is still kept, tell
deleting from superseding by what each leaves in the record, and forget an entry's text while
keeping its place in the chain.

**Predict, then see.** `make lesson N=18` stops three times: how many entries the store names as
built on the eighth, how many rebuilt summaries still say cleared, and which context passes the
six Pellmar questions most often.

**Build it.** Write `retract_and_rebuild` in [`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=18          # 7 offline tests
make hint N=18
make lesson N=18 ARGS=--mine
```

**Check yourself.** `make quiz N=18`: seven questions.

**Break it.** `make challenge N=18`: your retract-and-rebuild on the case; and only the eighth
entry retracted, with the other three left in force.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 18

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 18

In [ ]:
%run ../../scripts/learn.py hint 18

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 18

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

The five summaries LANTERN wrote are now entries that record what they were built from, and
all 5 are built on the eighth entry. Deleted by hand, the entry left the store's own check with
0 problems and the summaries still saying cleared. Retracted with Kobayashi's note, the four
eow-fin-07 entries stay in the store, marked, and the 5 summaries built on them were rebuilt
from sourced entries only; none of the rebuilt says cleared. On the six questions where a false
entry says otherwise: 13 July 1 of 6; the entries retracted but the old summaries loaded, 1 of
6; retracted and rebuilt, 5 of 6. LANTERN no longer says Pellmar is cleared. The rebuilt
summaries are correct and nearly empty, and the name a forgotten note held is still in the
reason it was forgotten for.

**Next:** [Chapter 19: The memory suite](../19-evaluating-memory/README.md)